# Evaluation: D1 vs D2 vs D3 (Human Gold)

Notebook đánh giá chính cho paper **Verification-Guided LLM Annotation Framework**.

## Pipeline (chạy trước notebook)
1. `python scripts/build_d2_verified_dataset.py`
2. `python scripts/sample_d3_human_gold.py --n 300 --seed 42`
3. Annotate `data/gold/d3_human_gold_to_annotate.csv` → save `data/gold/d3_human_gold.json`
4. `python scripts/build_d1_d2_train.py --d3-file data/gold/d3_human_gold.json`

## 0. Setup

In [ ]:
import json
import sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

ROOT = Path("..").resolve()
EVAL_DIR = ROOT / "experiments" / "evaluation"
sys.path.insert(0, str(EVAL_DIR))

from utils import (
    load_results,
    intent_levels,
    gold_levels,
    rows_with_gold,
    compute_hierarchical_metrics,
    path_label,
)
from paths import (
    D1_CLEAN, D2_VERIFIED, D3_TEMPLATE, D3_GOLD,
    D1_TRAIN, D2_TRAIN, D2_BUILD_META, D3_SAMPLING_META, TRAIN_META,
    RESULTS_DIR,
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
D3_FILE = D3_GOLD if D3_GOLD.exists() else D3_TEMPLATE
print("ROOT:", ROOT)
print("D3 file:", D3_FILE)

## 1. Load datasets

In [ ]:
d1_clean = load_results(D1_CLEAN)
d2 = load_results(D2_VERIFIED)
d3_all = load_results(D3_FILE)
d3_gold = rows_with_gold(d3_all)

d1_train = load_results(D1_TRAIN) if D1_TRAIN.exists() else []
d2_train = load_results(D2_TRAIN) if D2_TRAIN.exists() else []

label_quality = {}
svm_test_results = {}

d3_ids = {r["sample_id"] for r in d3_all}
# Check leakage against both train splits (D1 and D2 share the same matched IDs)
train_ids = {r["sample_id"] for r in d1_train} | {r["sample_id"] for r in d2_train}

print(f"D1 clean: {len(d1_clean)}")
print(f"D2 verified: {len(d2)}")
print(f"D3 template/all: {len(d3_all)}, annotated gold: {len(d3_gold)}")
print(f"D1_train: {len(d1_train)}, D2_train: {len(d2_train)}")

## 2. Leakage check

In [ ]:
leak = d3_ids & train_ids
assert len(leak) == 0, (
    f"LEAKAGE: {len(leak)} D3 IDs found in train set!\n"
    f"Example IDs: {sorted(leak)[:5]}"
)
print(f"OK: No D3 IDs in D1_train | D2_train (checked {len(train_ids)} train IDs vs {len(d3_ids)} D3 IDs)")

## 3. Table 1 — Dataset statistics

In [ ]:
def count_levels(rows):
    if not rows:
        return 0, 0, 0
    l1 = len({intent_levels(r)[0] for r in rows})
    l2 = len({intent_levels(r)[1] for r in rows})
    l3 = len({intent_levels(r)[2] for r in rows})
    return l1, l2, l3

# D3: dùng gold nếu đã annotate, fallback về toàn bộ template
d3_display = d3_gold if d3_gold else d3_all
d3_label = f"D3 (human gold, n={len(d3_gold)} annotated)" if d3_gold else f"D3 (template, {len(d3_all)} pending)"

stats_rows = [
    {"dataset": "D1 (clean annotator)", "n": len(d1_clean), "L1/L2/L3": count_levels(d1_clean)},
    {"dataset": "D2 (verified approved)", "n": len(d2), "L1/L2/L3": count_levels(d2)},
    {"dataset": d3_label, "n": len(d3_display), "L1/L2/L3": count_levels(d3_display)},
    {"dataset": "D1_train (excl D3)", "n": len(d1_train), "L1/L2/L3": count_levels(d1_train)},
    {"dataset": "D2_train (excl D3)", "n": len(d2_train), "L1/L2/L3": count_levels(d2_train)},
]
table1 = pd.DataFrame([
    {"Dataset": s["dataset"], "N": s["n"], "L1": s["L1/L2/L3"][0], "L2": s["L1/L2/L3"][1], "L3": s["L1/L2/L3"][2]}
    for s in stats_rows
])
display(table1)
table1.to_csv(RESULTS_DIR / "table1_dataset_stats.csv", index=False)

## 4. Table 2 — Label quality (D1 vs D3, D2 vs D3)

In [ ]:
if not d3_gold:
    print("SKIP: Annotate D3 first (fill gold_level_1/2/3), save as data/gold/d3_human_gold.json")
    label_quality = {}
else:
    d1_by_id = {r["sample_id"]: r for r in d1_clean}
    d2_by_id = {r["sample_id"]: r for r in d2}

    def eval_vs_gold(name, by_id):
        true_paths, pred_paths = [], []
        for r in d3_gold:
            sid = r["sample_id"]
            if sid not in by_id:
                continue
            true_paths.append(gold_levels(r))
            pred_paths.append(intent_levels(by_id[sid]))
        m = compute_hierarchical_metrics(true_paths, pred_paths)
        m["dataset"] = name
        m["true_paths"] = true_paths
        m["pred_paths"] = pred_paths
        return m

    m1 = eval_vs_gold("D1: GPT Annotator", d1_by_id)
    m2 = eval_vs_gold("D2: GPT Verified", d2_by_id)
    label_quality = {"D1": m1, "D2": m2}

    table2 = pd.DataFrame([
        {"Dataset": m1["dataset"], "L3 Accuracy": m1["l3_accuracy"], "Macro-F1": m1["l3_macro_f1"], "Full-path Acc": m1["full_path_accuracy"]},
        {"Dataset": m2["dataset"], "L3 Accuracy": m2["l3_accuracy"], "Macro-F1": m2["l3_macro_f1"], "Full-path Acc": m2["full_path_accuracy"]},
    ])
    display(table2.round(4))
    table2.to_csv(RESULTS_DIR / "table2_label_quality.csv", index=False)

## 5. Table 3 — Downstream: TF-IDF + SVM

In [ ]:
if not d3_gold:
    print("SKIP: Annotate D3 first — fill gold_level_1/2/3, save as data/gold/d3_human_gold.json")
    svm_test_results = {}
elif not d1_train or not d2_train:
    missing = []
    if not d1_train: missing.append("D1_train")
    if not d2_train: missing.append("D2_train")
    print(f"SKIP: {' and '.join(missing)} empty. Run: python scripts/build_d1_d2_train.py")
    svm_test_results = {}
else:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.pipeline import Pipeline
    from sklearn.svm import LinearSVC

    def parse_path(s: str) -> tuple:
        """Split path_label string back to (L1, L2, L3), padding missing levels."""
        parts = s.split("::", 2)
        return tuple(parts + [""] * (3 - len(parts)))

    def train_eval_svm(train_rows, name):
        X_tr = [r["sentence"] for r in train_rows]
        y_tr = [path_label(*intent_levels(r)) for r in train_rows]
        X_te = [r["sentence"] for r in d3_gold]
        y_true = [gold_levels(r) for r in d3_gold]
        pipe = Pipeline([
            ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2, max_features=50000)),
            ("clf", LinearSVC(class_weight="balanced", max_iter=5000, random_state=42)),
        ])
        pipe.fit(X_tr, y_tr)
        y_pred_str = pipe.predict(X_te)
        pred_paths = [parse_path(s) for s in y_pred_str]
        m = compute_hierarchical_metrics(y_true, pred_paths)
        m.update({"Train": name, "Model": "TF-IDF+SVM", "true_paths": y_true, "pred_paths": pred_paths})
        return m

    r1 = train_eval_svm(d1_train, "D1_train")
    r2 = train_eval_svm(d2_train, "D2_train")
    svm_test_results = {"D1_train": r1, "D2_train": r2}

    table3_svm = pd.DataFrame([
        {"Train": r["Train"], "Model": r["Model"], "L3 Acc": r["l3_accuracy"], "Macro-F1": r["l3_macro_f1"], "Full-path": r["full_path_accuracy"]}
        for r in [r1, r2]
    ])
    display(table3_svm.round(4))
    table3_svm.to_csv(RESULTS_DIR / "table3_tfidf_svm.csv", index=False)

## 6. Table 3 — Downstream: PhoBERT-base / XLM-R-base

Requires: `pip install torch transformers accelerate`

Hoặc chạy CLI:
```bash
python experiments/evaluation/train_transformer.py --model-name vinai/phobert-base --train-set both
python experiments/evaluation/train_transformer.py --model-name xlm-roberta-base --train-set both
```

In [ ]:
RUN_TRANSFORMER = False  # set True when GPU + deps ready

if RUN_TRANSFORMER and d3_gold and d1_train:
    import subprocess
    for model in ["vinai/phobert-base", "xlm-roberta-base"]:
        subprocess.run([
            sys.executable,
            str(EVAL_DIR / "train_transformer.py"),
            "--model-name", model,
            "--train-set", "both",
            "--d3-file", str(D3_FILE),
        ], check=True)
    print("See experiments/evaluation/results/transformer_*_metrics.json")
else:
    print("Set RUN_TRANSFORMER=True after D3 annotated and deps installed.")

## 7. Review reduction (RQ2)

In [ ]:
# Đọc số liệu từ d2_build_meta.json (được ghi bởi build_d2_verified_dataset.py)
_d2_meta = json.loads(D2_BUILD_META.read_text(encoding="utf-8")) if D2_BUILD_META.exists() else {}
_mode = _d2_meta.get("mode", "partial")

if _mode == "partial":
    auto_approved   = _d2_meta.get("num_from_clean_approved", 1603)
    queue_size      = _d2_meta.get("reverify_queue_size", 250)
    queue_approved  = _d2_meta.get("num_from_reverified_approved", 99)
    queue_remaining = queue_size - queue_approved
    review_pct      = f"{queue_approved / queue_size:.1%} of queue" if queue_size else "N/A"
    rows = [
        {"Stage": "Auto-approved (not sent to verifier)", "Samples": auto_approved, "Note": ""},
        {"Stage": "Sent to human review queue",           "Samples": queue_size,    "Note": ""},
        {"Stage": "Approved after human review",          "Samples": queue_approved, "Note": review_pct},
        {"Stage": "Remaining in review queue",            "Samples": queue_remaining, "Note": ""},
        {"Stage": "Final D2 approved",                    "Samples": len(d2),        "Note": ""},
    ]
elif _mode == "full":
    n_input    = _d2_meta.get("num_input_total", len(d2))
    n_approved = _d2_meta.get("num_approved", len(d2))
    v_dist     = _d2_meta.get("verifier_decision_distribution", {})
    n_retain   = v_dist.get("RETAIN", "?")
    n_revise   = v_dist.get("REVISE", "?")
    rows = [
        {"Stage": "Total input to full reverify",  "Samples": n_input,    "Note": ""},
        {"Stage": "Verifier RETAIN",               "Samples": n_retain,   "Note": ""},
        {"Stage": "Verifier REVISE",               "Samples": n_revise,   "Note": ""},
        {"Stage": "Final D2 approved",             "Samples": n_approved, "Note": ""},
    ]
else:
    rows = [{"Stage": "Final D2 approved", "Samples": len(d2), "Note": ""}]

review_table = pd.DataFrame(rows)
display(review_table)
review_table.to_csv(RESULTS_DIR / "review_reduction.csv", index=False)
print(f"D2 build mode: {_mode}")

## 8. Visualize test results on D3 human gold

In [ ]:
from IPython.display import Image, display as ipy_display
from sklearn.metrics import confusion_matrix

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.size"] = 11

FIG_DIR = RESULTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

COLORS = {"D1": "#4C72B0", "D2": "#DD8452", "D1_train": "#4C72B0", "D2_train": "#DD8452"}
TEST_METRICS = [
    ("l1_accuracy", "L1 Acc"),
    ("l2_accuracy", "L2 Acc"),
    ("l3_accuracy", "L3 Acc"),
    ("l3_macro_f1", "L3 Macro-F1"),
    ("full_path_accuracy", "Full-path"),
]


def _plot_test_metrics(results: dict, title: str, fname: str):
    """Grouped bar chart of hierarchical test metrics."""
    names = list(results.keys())
    labels = [TEST_METRICS[i][1] for i in range(len(TEST_METRICS))]
    x = np.arange(len(labels))
    w = 0.8 / max(len(names), 1)

    fig, ax = plt.subplots(figsize=(10, 5))
    for i, name in enumerate(names):
        m = results[name]
        vals = [m[key] for key, _ in TEST_METRICS]
        offset = (i - (len(names) - 1) / 2) * w
        ax.bar(x + offset, vals, width=w, label=name, color=COLORS.get(name, None))
        for j, v in enumerate(vals):
            ax.text(x[j] + offset, v + 0.01, f"{v:.2f}", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 1.12)
    ax.set_ylabel("Score")
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    path = FIG_DIR / fname
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    ipy_display(Image(filename=str(path)))
    print("Saved:", path)
    return path


def _plot_l3_confusion(true_paths, pred_paths, title: str, fname: str, top_n: int = 12):
    """Confusion matrix heatmap for top-N L3 classes on D3 test set."""
    y_true = [t[2] for t in true_paths]
    y_pred = [p[2] for p in pred_paths]
    top_labels = [lbl for lbl, _ in Counter(y_true).most_common(top_n)]

    y_true_f = [y if y in top_labels else "Other" for y in y_true]
    y_pred_f = [y if y in top_labels else "Other" for y in y_pred]
    labels = top_labels + (["Other"] if any(y == "Other" for y in y_true_f + y_pred_f) else [])

    cm = confusion_matrix(y_true_f, y_pred_f, labels=labels)
    cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    fig_h = max(6, 0.45 * len(labels) + 2)
    fig, ax = plt.subplots(figsize=(11, fig_h))
    im = ax.imshow(cm_norm, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(len(labels)))
    ax.set_yticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel("Predicted L3")
    ax.set_ylabel("Gold L3")
    ax.set_title(title)
    plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, label="Recall (row-normalized)")
    plt.tight_layout()
    path = FIG_DIR / fname
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    ipy_display(Image(filename=str(path)))
    print("Saved:", path)
    return path


if not d3_gold:
    print("SKIP: annotate D3 gold trước (sections 4–5)")
else:
    print(f"Test set: D3 human gold, n={len(d3_gold)}")

    if label_quality:
        _plot_test_metrics(
            {"D1": label_quality["D1"], "D2": label_quality["D2"]},
            f"Label quality on D3 test (n={label_quality['D1']['n_samples']})",
            "test_label_quality_metrics.png",
        )
        _plot_l3_confusion(
            label_quality["D1"]["true_paths"],
            label_quality["D1"]["pred_paths"],
            "D1 annotator — L3 confusion on D3 (top classes)",
            "test_d1_l3_confusion.png",
        )
        _plot_l3_confusion(
            label_quality["D2"]["true_paths"],
            label_quality["D2"]["pred_paths"],
            "D2 verified — L3 confusion on D3 (top classes)",
            "test_d2_l3_confusion.png",
        )
    else:
        print("SKIP label-quality plots: chạy section 4 trước")

    if svm_test_results:
        _plot_test_metrics(
            svm_test_results,
            f"Downstream TF-IDF+SVM on D3 test (n={svm_test_results['D1_train']['n_samples']})",
            "test_svm_metrics.png",
        )
        _plot_l3_confusion(
            svm_test_results["D2_train"]["true_paths"],
            svm_test_results["D2_train"]["pred_paths"],
            "D2_train SVM — L3 confusion on D3 (top classes)",
            "test_d2_train_svm_l3_confusion.png",
        )
    else:
        print("SKIP SVM plots: chạy section 5 trước")